# 01 · 정규화 edit distance와 읽기 순서

OmniDocBench 평가의 두 기초 신호를 표준 라이브러리로 구현한다. 공식 evaluator나 논문 점수의 재현이 아닌 toy reproduction이다.

**학습 목표**: Unicode 정규화, normalized edit distance와 reading-order error의 계산·해석을 익힌다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행한다. 표준 라이브러리 `unicodedata`만 사용하며 외부 패키지는 없다.

In [ ]:
# NFKC는 전각 문자처럼 모양은 비슷하지만 code point가 다른 표현을 먼저 통일한다.
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    return ' '.join(text.split())

def levenshtein(a, b):
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]

def normalized_edit(reference, prediction):
    ref, pred = normalize_text(reference), normalize_text(prediction)
    return levenshtein(ref, pred) / max(1, len(ref), len(pred))

def order_error(reference_ids, predicted_ids):
    return levenshtein(reference_ids, predicted_ids) / max(1, len(reference_ids), len(predicted_ids))


In [ ]:
pairs = [('Net income: 42M', 'Net  income: 42M'), ('formula x²', 'formula x2')]
for reference, prediction in pairs:
    print(reference, 'vs', prediction, '=>', round(normalized_edit(reference, prediction), 3))

ref_order = ['title', 'left-1', 'left-2', 'right-1']
pred_order = ['title', 'right-1', 'left-1', 'left-2']
print('reading-order error:', order_error(ref_order, pred_order))
assert normalized_edit('Ａ  B', 'a b') == 0.0
assert order_error(ref_order, ref_order) == 0.0


정규화 규칙이 점수의 일부다. 실제 비교에서는 Unicode, 공백, 수식과 Markdown 처리 순서를 evaluator revision과 함께 고정해야 한다.